# AusLAMP Victoria: Quick Start Demo

**Goal**: Demonstrate the new unified MTH5 reader for both EDL and LEMI-424 data.

**Key Features**:
- Single reader for both instrument types
- Automatic instrument detection and calibration
- Read directly from GA's MTH5 files (no ASCII conversion needed)
- Correct Bartington Mag-03 calibration applied

**Data Source**: `E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_MTH5\` and `LEMI_MTH5\`

In [ ]:
import sys
sys.path.insert(0, '../src')

from readers import read_station, get_station_metadata, get_all_metadata
from config import DataPaths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Read EDL Station (VIC001)

EDL data:
- Raw data in µV (uncalibrated in MTH5)
- Reader applies correct Bartington Mag-03 calibration
- Sample rate: 10 Hz

In [ ]:
# Get metadata first
edl_file = DataPaths.EDL_MTH5 / 'VIC001.h5'
metadata_edl = get_station_metadata(edl_file)

print("VIC001 Metadata:")
for key, value in metadata_edl.items():
    print(f"  {key}: {value}")

In [ ]:
# Read data (load first hour for quick demo)
df_edl = read_station(edl_file, channels='magnetic', verbose=True)
df_edl_sample = df_edl.iloc[:36000]  # First hour at 10 Hz

print("\nSample data:")
df_edl_sample.head()

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for i, ch in enumerate(['BX', 'BY', 'BZ']):
    axes[i].plot(df_edl_sample.index, df_edl_sample[ch], linewidth=0.5)
    axes[i].set_ylabel(f'{ch} (nT)', fontsize=11)
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('VIC001 (EDL) - First Hour', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Time', fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(df_edl_sample.describe())

## 2. Read LEMI-424 Station (VIC065R)

LEMI-424 data:
- Already calibrated in nT
- Sample rate: 1 Hz
- Multiple runs (long deployment with battery swaps)

In [ ]:
# Get metadata
lemi_file = DataPaths.LEMI_MTH5 / 'VIC065R.h5'
metadata_lemi = get_station_metadata(lemi_file)

print("VIC065R Metadata:")
for key, value in metadata_lemi.items():
    print(f"  {key}: {value}")

In [ ]:
# Read data (load first 24 hours)
df_lemi = read_station(lemi_file, channels='magnetic', verbose=True)
df_lemi_sample = df_lemi.iloc[:86400]  # First 24 hours at 1 Hz

print("\nSample data:")
df_lemi_sample.head()

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for i, ch in enumerate(['BX', 'BY', 'BZ']):
    axes[i].plot(df_lemi_sample.index, df_lemi_sample[ch], linewidth=0.5)
    axes[i].set_ylabel(f'{ch} (nT)', fontsize=11)
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('VIC065R (LEMI-424) - First 24 Hours', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Time', fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(df_lemi_sample.describe())

## 3. Compare Field Strengths

Calculate horizontal and total field for both stations.

In [ ]:
# Calculate derived quantities
for df, name in [(df_edl_sample, 'VIC001 (EDL)'), (df_lemi_sample, 'VIC065R (LEMI-424)')]:
    h_field = np.sqrt(df['BX']**2 + df['BY']**2)
    f_field = np.sqrt(df['BX']**2 + df['BY']**2 + df['BZ']**2)

    print(f"\n{name}:")
    print(f"  BX: {df['BX'].mean():.2f} ± {df['BX'].std():.2f} nT")
    print(f"  BY: {df['BY'].mean():.2f} ± {df['BY'].std():.2f} nT")
    print(f"  BZ: {df['BZ'].mean():.2f} ± {df['BZ'].std():.2f} nT")
    print(f"  H (horizontal): {h_field.mean():.2f} nT")
    print(f"  F (total): {f_field.mean():.2f} nT")

## 4. Load All Metadata

Get metadata for all sites without loading full time series.

In [ ]:
# Load all EDL metadata
print("Loading EDL metadata...")
df_edl_meta = get_all_metadata(DataPaths.EDL_MTH5)

# Load all LEMI metadata
print("\nLoading LEMI-424 metadata...")
df_lemi_meta = get_all_metadata(DataPaths.LEMI_MTH5)

# Combine
df_all_meta = pd.concat([df_edl_meta, df_lemi_meta], ignore_index=True)

print(f"\nTotal sites: {len(df_all_meta)}")
print(f"  EDL: {len(df_edl_meta)}")
print(f"  LEMI-424: {len(df_lemi_meta)}")

df_all_meta.head(10)

In [ ]:
# Summary by instrument
print("\nRecording Duration Summary:")
print(df_all_meta.groupby('instrument')['duration_days'].describe())

## Summary

The new unified MTH5 reader:

✅ **Works with both EDL and LEMI-424** automatically  
✅ **Applies correct calibration** (Bartington Mag-03 for EDL)  
✅ **Returns consistent DataFrame format**  
✅ **Handles multiple runs** (LEMI-424 long deployments)  
✅ **Fast metadata extraction** without loading full datasets  

**Next steps**:
1. Build SQLite metadata database for all 100 sites
2. QA/QC analysis with the new reader
3. Site location maps and timeline plots
4. MT processing workflow